# Finance assistant: Temporal Serverless Workers on AgentCore

This notebook deploys the finance Temporal Worker to Amazon Bedrock AgentCore Runtime and registers its **named Runtime endpoint** as a Temporal Serverless Worker compute provider. The finance Workflow is still started and signaled through a Temporal Client. Temporal starts Worker capacity when the task queue needs it.

AgentCore Serverless Workers are a Temporal Cloud Pre-release feature. Request access for an AWS-hosted namespace before deployment. See the [Temporal deployment guide](https://docs.temporal.io/production-deployment/worker-deployments/serverless-workers/agentcore). The Runtime and its named endpoint use AWS IAM inbound authorization so Temporal can invoke them through an assumed role.


## Prerequisites

1. From this directory, run `uv sync --python 3.12` and `uv pip install --python .venv/bin/python jupyterlab ipykernel`, then start `.venv/bin/jupyter lab` here and use the project Python kernel.
2. Install Temporal CLI v1.8.3 or newer, configure access to your Temporal Cloud namespace, and make `temporal` available to the kernel process.
3. Configure AWS credentials for an account and AgentCore-supported Region, including permission to deploy AgentCore resources, ECR, IAM, CloudFormation, and Secrets Manager. Set `AWS_PROFILE` if applicable, then authenticate outside the notebook.
4. Copy `.env.example` to `.env` in this directory. Fill in the Temporal Cloud address, namespace, API key, a **new** build ID for this deployment run, and an External ID of at least five characters. Run the notebook from this directory.

Do not reuse a build ID when rerunning deployment. Each build ID gets a named endpoint pinned to one immutable Runtime version. The notebook will reject an existing endpoint that targets another version.


In [ ]:
import json
import os
import re
import subprocess
import time
from pathlib import Path

import boto3
from botocore.exceptions import ClientError
from dotenv import load_dotenv
from bedrock_agentcore_starter_toolkit import Runtime

assert Path.cwd().name == "finance-personal-assistant", "Open Jupyter from the finance-personal-assistant directory"
load_dotenv(".env", override=True)

required = (
    "TEMPORAL_ADDRESS",
    "TEMPORAL_NAMESPACE",
    "TEMPORAL_API_KEY",
    "TEMPORAL_BUILD_ID",
    "TEMPORAL_INVOKE_EXTERNAL_ID",
)
missing = [name for name in required if not os.environ.get(name)]
if missing:
    raise RuntimeError(f"Set these variables in .env: {', '.join(missing)}")
placeholders = [name for name in required if "<" in os.environ[name] or ">" in os.environ[name]]
if placeholders:
    raise RuntimeError(f"Replace placeholder values in .env: {', '.join(placeholders)}")

build_id = os.environ["TEMPORAL_BUILD_ID"]
if not re.fullmatch(r"[A-Za-z0-9_-]{1,32}", build_id):
    raise ValueError("TEMPORAL_BUILD_ID must be 1-32 letters, digits, underscores, or hyphens")
external_id = os.environ["TEMPORAL_INVOKE_EXTERNAL_ID"]
if not re.fullmatch(r"[A-Za-z0-9_+=,.@-]{5,45}", external_id):
    raise ValueError("TEMPORAL_INVOKE_EXTERNAL_ID must be 5-45 supported IAM External ID characters")

session = boto3.Session()
region = os.environ.get("AWS_REGION") or session.region_name
if not region:
    raise RuntimeError("Set AWS_REGION or configure a default AWS Region")
account_id = session.client("sts", region_name=region).get_caller_identity()["Account"]
namespace = os.environ["TEMPORAL_NAMESPACE"]
endpoint_name = "temporal_" + build_id.replace("-", "_")
print(f"AWS account: {account_id}; Region: {region}; Temporal namespace: {namespace}; build: {build_id}")


## 1. Store the Temporal API key

The key stays out of AgentCore Runtime environment variables. This cell creates a Secrets Manager secret or rotates its value if it already exists. The Runtime execution role receives read access after deployment.


In [ ]:
secrets = session.client("secretsmanager", region_name=region)
secret_name = f"finance-temporal-api-key-{namespace.replace('.', '-') }"
try:
    secret = secrets.describe_secret(SecretId=secret_name)
except secrets.exceptions.ResourceNotFoundException:
    secret = secrets.create_secret(
        Name=secret_name, SecretString=os.environ["TEMPORAL_API_KEY"]
    )
else:
    secrets.put_secret_value(
        SecretId=secret_name, SecretString=os.environ["TEMPORAL_API_KEY"]
    )
secret_arn = secret["ARN"]
print(f"Temporal API key secret: {secret_arn}")


## 2. Deploy the Worker Runtime

The existing AgentCore starter toolkit builds the finance project as a container and creates the Runtime execution role. Omitting a custom JWT authorizer uses AgentCore's default IAM inbound authorization. The Worker reports `HealthyBusy` while it polls and drains after 60 seconds without Activity work.


In [ ]:
agentcore_runtime = Runtime()
agentcore_runtime.configure(
    entrypoint="worker.py",
    agent_name="temporal_orchestrated_financial_assistant",
    requirements_file="requirements.txt",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    region=region,
    protocol="HTTP",
    idle_timeout=900,
    max_lifetime=28800,
)
print("AgentCore Runtime configured")


In [ ]:
launch_result = agentcore_runtime.launch(
    auto_update_on_conflict=True,
    env_vars={
        "TEMPORAL_ADDRESS": os.environ["TEMPORAL_ADDRESS"],
        "TEMPORAL_NAMESPACE": namespace,
        "TEMPORAL_BUILD_ID": build_id,
        "TEMPORAL_API_KEY_SECRET_ARN": secret_arn,
        "AWS_REGION": region,
        "AWS_DEFAULT_REGION": region,
        "AGENTCORE_DEBOUNCE_SECONDS": "60",
        "MPLBACKEND": "Agg",
        "OTEL_PYTHON_EXCLUDED_URLS": "/ping,/invocations",
    },
)
runtime_id = launch_result.agent_id
runtime_arn = launch_result.agent_arn
print(f"AgentCore Runtime ARN: {runtime_arn}")


## 3. Grant the Runtime access to its API key and Bedrock

The execution role runs the finance Activities. Temporal uses a different invocation role created below.


In [ ]:
control = session.client("bedrock-agentcore-control", region_name=region)
runtime = control.get_agent_runtime(agentRuntimeId=runtime_id)
if runtime["status"] != "READY":
    raise RuntimeError(f"AgentCore Runtime is not ready: {runtime['status']}")
runtime_version = runtime["agentRuntimeVersion"]
execution_role_name = runtime["roleArn"].rsplit("/", 1)[-1]
iam = session.client("iam")
iam.put_role_policy(
    RoleName=execution_role_name,
    PolicyName="FinanceTemporalWorkerAccess",
    PolicyDocument=json.dumps({
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": "secretsmanager:GetSecretValue",
                "Resource": secret_arn,
            },
            {
                "Effect": "Allow",
                "Action": [
                    "bedrock:InvokeModel",
                    "bedrock:InvokeModelWithResponseStream",
                ],
                "Resource": "*",
            },
        ],
    }),
)
print(f"Runtime version {runtime_version}; execution role permissions updated")


## 4. Create a named endpoint for this Worker version

Temporal needs the **endpoint ARN**, which includes `/runtime-endpoint/<name>`. The Runtime ARN alone is insufficient. An existing endpoint must continue pointing to the same Runtime version so pinned Workflows can replay safely.


In [ ]:
try:
    endpoint = control.get_agent_runtime_endpoint(
        agentRuntimeId=runtime_id, endpointName=endpoint_name
    )
except control.exceptions.ResourceNotFoundException:
    endpoint = control.create_agent_runtime_endpoint(
        agentRuntimeId=runtime_id,
        name=endpoint_name,
        agentRuntimeVersion=runtime_version,
        description=f"Temporal finance Worker {build_id}",
    )
else:
    if endpoint["liveVersion"] != runtime_version:
        raise RuntimeError(
            f"{endpoint_name} already points at Runtime version "
            f"{endpoint['liveVersion']}; use a new TEMPORAL_BUILD_ID for new code"
        )

for _ in range(36):
    endpoint = control.get_agent_runtime_endpoint(
        agentRuntimeId=runtime_id, endpointName=endpoint_name
    )
    if endpoint["status"] == "READY":
        break
    if endpoint["status"] in {"CREATE_FAILED", "UPDATE_FAILED"}:
        raise RuntimeError(f"AgentCore endpoint failed: {endpoint['status']}")
    time.sleep(5)
else:
    raise TimeoutError("AgentCore endpoint was not ready within three minutes")
endpoint_arn = endpoint["agentRuntimeEndpointArn"]
print(f"Named endpoint ARN: {endpoint_arn}")


## 5. Allow Temporal Cloud to invoke the endpoint

The CloudFormation template is based on Temporal's [AgentCore invocation role template](https://docs.temporal.io/files/temporal-cloud-serverless-worker-agentcore-role.yaml). Its trust policy uses the External ID from `.env`, and its permissions are scoped to this Runtime and its endpoints.


In [ ]:
cloudformation = session.client("cloudformation", region_name=region)
stack_name = "finance-temporal-agentcore-invoke"
template = Path("temporal/cloud_agentcore_invoke_role.yaml").read_text()
parameters = [
    {"ParameterKey": "AssumeRoleExternalId", "ParameterValue": external_id},
    {"ParameterKey": "AgentRuntimeArn", "ParameterValue": runtime_arn},
]
try:
    existing_stack = cloudformation.describe_stacks(StackName=stack_name)["Stacks"][0]
except ClientError as exc:
    if "does not exist" not in str(exc):
        raise
    cloudformation.create_stack(
        StackName=stack_name,
        TemplateBody=template,
        Parameters=parameters,
        Capabilities=["CAPABILITY_NAMED_IAM"],
    )
    cloudformation.get_waiter("stack_create_complete").wait(StackName=stack_name)
else:
    old_parameters = {p["ParameterKey"]: p["ParameterValue"] for p in existing_stack["Parameters"]}
    wanted = {p["ParameterKey"]: p["ParameterValue"] for p in parameters}
    if old_parameters != wanted:
        raise RuntimeError(
            f"CloudFormation stack {stack_name} is scoped to another Runtime or External ID. "
            "Review and update that stack before continuing."
        )

stack = cloudformation.describe_stacks(StackName=stack_name)["Stacks"][0]
invoke_role_arn = next(o["OutputValue"] for o in stack["Outputs"] if o["OutputKey"] == "RoleArn")
print(f"Temporal Cloud invocation role: {invoke_role_arn}")


## 6. Register the Serverless Worker in Temporal

The Temporal CLI inherits `TEMPORAL_ADDRESS`, `TEMPORAL_NAMESPACE`, and `TEMPORAL_API_KEY` from `.env`. The Worker Deployment Version must use exactly the build ID sent to the Runtime. This cell refuses to overwrite an existing version pointing at a different endpoint.


In [ ]:
deployment_name = "finance-personal-assistant"

def temporal_cli(*args, check=True):
    result = subprocess.run(
        ["temporal", *args], capture_output=True, text=True, env=os.environ.copy()
    )
    if check and result.returncode:
        raise RuntimeError(result.stderr or result.stdout)
    return result

existing_deployment = temporal_cli(
    "worker", "deployment", "describe", "--name", deployment_name, check=False
)
if existing_deployment.returncode:
    temporal_cli("worker", "deployment", "create", "--name", deployment_name)

existing_version = temporal_cli(
    "worker", "deployment", "describe-version",
    "--deployment-name", deployment_name, "--build-id", build_id,
    "--output", "json", check=False,
)
if existing_version.returncode:
    temporal_cli(
        "worker", "deployment", "create-version",
        "--deployment-name", deployment_name,
        "--build-id", build_id,
        "--aws-agentcore-endpoint-arn", endpoint_arn,
        "--aws-agentcore-assume-role-arn", invoke_role_arn,
        "--aws-agentcore-assume-role-external-id", external_id,
    )
elif endpoint_arn not in existing_version.stdout:
    raise RuntimeError("This Temporal build ID already uses a different endpoint; choose a new build ID")
print(f"Registered {deployment_name}/{build_id} with {endpoint_arn}")


In the Temporal Cloud UI, open the Worker Deployment Version and use **Actions → Validate Connection**. This checks that Temporal can assume the role, read the named endpoint, and invoke the Runtime. Resolve any validation error before setting this version current.


In [ ]:
temporal_cli(
    "worker", "deployment", "set-current-version",
    "--deployment-name", deployment_name,
    "--build-id", build_id,
    "--allow-no-pollers", "--yes",
)
print(f"Current Worker Deployment Version: {deployment_name}/{build_id}")


## 7. Try the finance Workflow

Run `.venv/bin/python -m temporal.start_workflow` from this directory. Do **not** invoke AgentCore manually. Temporal should invoke the named endpoint when the task queue needs a Worker. After the budget report is ready and the Worker has drained, signal an investment amount in the CLI; Temporal should start capacity again to finish the Workflow.

Check the Worker Deployment in Temporal Cloud and AgentCore Runtime logs to confirm the scale-from-zero start and later restart. See the README for the full verification sequence.
